In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

# Loading the finalized data
matches = pd.read_csv("international_matches_transformed.csv")
matches["date"] = pd.to_datetime(matches["date"])

# 2. Rolling Averages
# Using GF & GA
cols = ["gf", "ga"]
new_cols = [f"{c}_rolling" for c in cols]

def rolling_averages(group, cols, new_cols):
    group = group.sort_values("date")
    rolling_stats = group[cols].rolling(3, closed='left').mean()
    group[new_cols] = rolling_stats
    return group.dropna(subset=new_cols)

matches_rolling = matches.groupby("team").apply(lambda x: rolling_averages(x, cols, new_cols))
matches_rolling = matches_rolling.droplevel('team')

# 3. Matchup History: Head-to-Head (H2H) Win Rate
# This captures if a team "normally loses" to a specific opponent
def get_h2h_win_rate(group):
    group = group.sort_values("date")
    # Cumulative win rate against this specific opponent until the match day
    h2h_rate = group['target'].expanding().mean().shift(1)
    group['h2h_win_rate'] = h2h_rate.fillna(0.5)
    return group

# Create unique matchup key (e.g., 'Brazil-France')
matches_rolling['matchup'] = matches_rolling.apply(lambda x: "-".join(sorted([x['team'], x['opponent']])), axis=1)
matches_rolling = matches_rolling.groupby(['team', 'matchup']).apply(get_h2h_win_rate)
matches_rolling = matches_rolling.droplevel(['team', 'matchup'])

# 4. Weighting: Tournament Importance & Time Decay
def calculate_weights(row):
    # Tournament Importance Score
    tourney_weights = {
        'FIFA World Cup': 5.0,
        'UEFA Euro': 3.5,
        'Copa América': 3.5,
        'Friendly': 1.0
    }
    t_weight = tourney_weights.get(row['tournament'], 2.0)

    # Time Decay: Exponentially lower weight for older matches
    # Formula: w = e^{-0.05 * years_ago}
    years_ago = 2026 - row['date'].year
    time_weight = np.exp(-0.05 * years_ago)

    return t_weight * time_weight

matches_rolling['sample_weight'] = matches_rolling.apply(calculate_weights, axis=1)

# 5. Model Training with Weights
predictors = ["venue_code", "opp_code", "day_code", "h2h_win_rate"]
all_features = predictors + new_cols

# Split: Train on everything before 2022, test on recent results
train = matches_rolling[matches_rolling["date"] < '2022-01-01']
test = matches_rolling[matches_rolling["date"] >= '2022-01-01']

rf = RandomForestClassifier(n_estimators=100, min_samples_split=10, random_state=1)

# Training using the calculated weights
rf.fit(train[all_features], train["target"], sample_weight=train["sample_weight"])

# 6. Prediction Evaluation
preds = rf.predict(test[all_features])
precision = precision_score(test["target"], preds)
print(f"Model Precision (Win Prediction): {precision:.2%}")

# View a sample of predicted matches
combined = pd.DataFrame(dict(actual=test["target"], predicted=preds), index=test.index)
combined = combined.merge(test[["date", "team", "opponent", "tournament"]], left_index=True, right_index=True)
# Results for Argentina
team_name = "Argentina"
team_results = combined[combined["team"] == team_name]

print(f"\nRecent Prediction Samples for {team_name}:")

print(team_results.tail(10))

/tmp/ipython-input-4136676350.py:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  matches_rolling = matches.groupby("team").apply(lambda x: rolling_averages(x, cols, new_cols))
/tmp/ipython-input-4136676350.py:35: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  matches_rolling = matches_rolling.groupby(['team', 'matchup']).apply(get_h2h_win_rate)


Model Precision (Win Prediction): 61.56%

Recent Prediction Samples for Argentina:
        actual  predicted       date       team              opponent  \
47811        1          0 2022-11-30  Argentina                Poland   
51007        1          1 2025-10-14  Argentina           Puerto Rico   
99071        0          1 2022-11-22  Argentina          Saudi Arabia   
98992        1          0 2022-11-16  Argentina  United Arab Emirates   
48746        0          1 2023-11-16  Argentina               Uruguay   
101588       1          0 2025-03-21  Argentina               Uruguay   
47065        1          1 2022-03-25  Argentina             Venezuela   
101103       0          1 2024-10-10  Argentina             Venezuela   
50704        1          1 2025-09-04  Argentina             Venezuela   
50912        1          1 2025-10-10  Argentina             Venezuela   

                 tournament  
47811             World Cup  
51007              Friendly  
99071             World